In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps in UTC
metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

log = pd.read_csv("log.csv")
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

error_logs = pd.read_csv("error_logs.csv")
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregates: for each (cmdb_id, kpi_name) compute count, p50, p95 across full series
metrics_agg = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(
        count='count',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .head(50)
)

# 2) trace.csv: unique (cmdb_id, trace_name) with count of records
trace_counts = (
    trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
)

# 3) log.csv: unique (cmdb_id, log_name) with count of records
log_counts = (
    log
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
)

# 4) error_logs.csv: counts per cmdb_id and up to 5 sample messages (first 5 by timestamp)
# Ensure messages are strings
error_logs['message'] = error_logs['message'].astype(str)

def sample_messages(df):
    msgs = df.sort_values('timestamp')['message'].head(5).tolist()
    return '; '.join(msgs)

error_summary = (
    error_logs
    .groupby('cmdb_id', dropna=False)
    .agg(
        count=('message', 'count'),
        sample_messages=('message', lambda s: sample_messages(error_logs.loc[s.index]))
    )
    .reset_index()
    .sort_values('count', ascending=False)
)

# Display compact results (DataFrames). Use variable names as output.
metrics_agg, trace_counts, log_counts, error_summary

```
Out[1]:
```
Summary of the aggregation results (compact):

- metric.csv (up to 50 rows shown): 50 (cmdb_id, kpi_name) groups returned. Many groups have 25 samples. Examples of top groups:
  - ts-voucher-service / socket — count 25, p50 = 4.0, p95 = 4.0
  - ts-admin-basic-info-service / cpu — count 25, p50 ≈ 0.3635, p95 ≈ 0.6747
  - ts-admin-basic-info-service / diskio — count 25, p50 ≈ 51,395, p95 ≈ 57,048
  - ts-config-service / mem — count 25, p50 ≈ 253,563,100, p95 ≈ 253,618,000
  - Many other services/kpis follow with similar full-series statistics.

- trace.csv (up to 50 rows shown): top trace groups (count = 25) are dominated by ts-travel2-service traces (duration_mean, duration_p95, error_rate, row_count) and many ts-admin-travel-service traces. All listed trace groups have 25 records.

- log.csv (up to 50 rows shown): top log groups are log.error_count and log.row_count for multiple services (e.g., ts-admin-basic-info-service, ts-admin-travel-service). Most shown counts are 25 (some are 24).

- error_logs.csv: No error log records were returned (empty summary). There are no sample error messages in the dataset.

Key takeaway: telemetry contains many metric and trace series with complete 25-sample series for multiple services; log aggregates exist for many services; there are no error log messages available in error_logs.csv. If you want, next steps could be: inspect specific high-percentile KPIs (e.g., high p95 latency/diskio/memory) or drill into traces for services that show elevated KPIs.

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count           p50           p95
366           ts-voucher-service      socket     25  4.000000e+00  4.000000e+00
0    ts-admin-basic-info-service         cpu     25  3.635217e-01  6.747261e-01
1    ts-admin-basic-info-service      diskio     25  5.139528e+04  5.704817e+04
2    ts-admin-basic-info-service  latency-50     25  1.584755e-02  1.680484e-02
3    ts-admin-basic-info-service  latency-90     25  2.320349e-02  2.352737e-02
..                           ...         ...    ...           ...           ...
310              ts-travel-mongo      diskio     25  3.763237e+05  4.506848e+05
71             ts-config-service      diskio     25  5.135453e+04  6.237124e+04
72             ts-config-service  latency-50     25  3.434157e-03  3.550867e-03
73             ts-config-service  latency-90     25  7.256493e-03  7.794752e-03
74             ts-config-service         mem     25  2.535631e+08  2.536180e+08

[50 rows x 5 columns],                      cmdb_id                                     trace_name  count
728       ts-travel2-service  trace.to_ts-order-other-service.duration_mean     25
729       ts-travel2-service   trace.to_ts-order-other-service.duration_p95     25
730       ts-travel2-service     trace.to_ts-order-other-service.error_rate     25
731       ts-travel2-service      trace.to_ts-order-other-service.row_count     25
732       ts-travel2-service        trace.to_ts-route-service.duration_mean     25
..                       ...                                            ...    ...
89   ts-admin-travel-service        trace.to_ts-travel-service.duration_p95     25
90   ts-admin-travel-service          trace.to_ts-travel-service.error_rate     25
91   ts-admin-travel-service           trace.to_ts-travel-service.row_count     25
92   ts-admin-travel-service      trace.to_ts-travel2-service.duration_mean     25
93   ts-admin-travel-service       trace.to_ts-travel2-service.duration_p95     25

[50 rows x 3 columns],                         cmdb_id         log_name  count
0   ts-admin-basic-info-service  log.error_count     25
1   ts-admin-basic-info-service    log.row_count     25
2       ts-admin-travel-service  log.error_count     25
3       ts-admin-travel-service    log.row_count     25
7          ts-assurance-service    log.row_count     25
..                          ...              ...    ...
18       ts-consign-price-mongo  log.error_count     24
26            ts-food-map-mongo  log.error_count     24
27            ts-food-map-mongo    log.row_count     24
23            ts-contacts-mongo    log.row_count     24
22            ts-contacts-mongo  log.error_count     24

[50 rows x 3 columns], Empty DataFrame
Columns: [cmdb_id, count, sample_messages]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing DataFrames: metric, trace, log (assumed already loaded and timestamp-parsed)
components = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Incident window (UTC)
start = pd.to_datetime('2024-01-23 15:11:51', utc=True)
end = pd.to_datetime('2024-01-23 15:41:51', utc=True)

# 1) Global P95 and P5 per (cmdb_id, kpi_name) using the full metric series but only for the five components
metrics_for_components = metric[metric['cmdb_id'].isin(components)]
p95p5 = (
    metrics_for_components
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05),
        count='count'
    )
    .reset_index()
    .sort_values(['count', 'cmdb_id', 'kpi_name'], ascending=[False, True, True])
)
# Keep compact
p95p5_df = p95p5[['cmdb_id', 'kpi_name', 'count', 'p95', 'p5']].copy()

# 2) Filter metric.csv to only rows for these components and timestamps inside the incident window
metric_window = metric[
    metric['cmdb_id'].isin(components) &
    (metric['timestamp'] >= start) &
    (metric['timestamp'] <= end)
].copy()

# 3) For each (cmdb_id, kpi_name) from step (1) present anomaly summary restricted to the incident window
rows = []
for _, row in p95p5_df.iterrows():
    cid = row['cmdb_id']
    kpi = row['kpi_name']
    p95_val = row['p95']
    p5_val = row['p5']
    # select windowed series for this group
    grp = metric_window[(metric_window['cmdb_id'] == cid) & (metric_window['kpi_name'] == kpi)]
    if grp.empty:
        anomaly_count = 0
        earliest_ts = pd.NaT
        extremal = pd.NA
    else:
        high_mask = grp['value'] > p95_val
        low_mask = grp['value'] <= p5_val
        mask = high_mask | low_mask
        anomaly_count = int(mask.sum())
        if anomaly_count == 0:
            earliest_ts = pd.NaT
            extremal = pd.NA
        else:
            earliest_ts = grp.loc[mask, 'timestamp'].min()
            # Choose extremal value: prefer max of high anomalies if any; else min of low anomalies
            if high_mask.any():
                extremal = grp.loc[high_mask, 'value'].max()
            else:
                extremal = grp.loc[low_mask, 'value'].min()
    rows.append((cid, kpi, float(p95_val), float(p5_val), anomaly_count, earliest_ts, extremal))

anomalies_df = pd.DataFrame(rows, columns=[
    'cmdb_id', 'kpi_name', 'p95', 'p5', 'anomaly_count', 'earliest_anomaly_timestamp', 'extremal_value_in_window'
])

# Keep only groups that had any anomalies; if none, return empty DataFrame-like summary per instruction
if anomalies_df['anomaly_count'].sum() == 0:
    anomalies_summary = anomalies_df.iloc[0:0][['cmdb_id','kpi_name','p95','p5','anomaly_count','earliest_anomaly_timestamp','extremal_value_in_window']]
else:
    anomalies_summary = (
        anomalies_df[anomalies_df['anomaly_count'] > 0]
        .sort_values('anomaly_count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )

# 4) Filter trace.csv and log.csv to these five components and the incident window and return counts
trace_window = trace[
    trace['cmdb_id'].isin(components) &
    (trace['timestamp'] >= start) &
    (trace['timestamp'] <= end)
].copy()

trace_window_counts = (
    trace_window
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

log_window = log[
    log['cmdb_id'].isin(components) &
    (log['timestamp'] >= start) &
    (log['timestamp'] <= end)
].copy()

log_window_counts = (
    log_window
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# 5) Identify missing components in metric/trace/log (based on full files)
present_in_metric = set(metric['cmdb_id'].unique())
present_in_trace = set(trace['cmdb_id'].unique())
present_in_log = set(log['cmdb_id'].unique())

missing_metric = [c for c in components if c not in present_in_metric]
missing_trace = [c for c in components if c not in present_in_trace]
missing_log = [c for c in components if c not in present_in_log]

missing_summary = pd.DataFrame([
    {'dataset': 'metric.csv', 'missing_components': ', '.join(missing_metric) or '(none)'},
    {'dataset': 'trace.csv',  'missing_components': ', '.join(missing_trace) or '(none)'},
    {'dataset': 'log.csv',    'missing_components': ', '.join(missing_log) or '(none)'}
])

# Final compact outputs
p95p5_df, anomalies_summary, trace_window_counts, log_window_counts, missing_summary

```
Out[2]:
```
Summary (straightforward):

Data availability
- metric.csv and trace.csv contain data for all five components.  
- log.csv is missing ts-train-service (present for the other four).

Global thresholds (full metric series, per (cmdb_id, kpi_name))
- Computed P95 and P5 for 37 (cmdb_id, kpi_name) groups across the five components (examples):
  - ts-auth-service: cpu P95 ≈ 39.73, P5 ≈ 13.66; diskio P95 ≈ 105,589; mem P95 ≈ 263,467,300
  - ts-order-service: socket P95 ≈ 12.8467, P5 ≈ 11.1167; mem P95 ≈ 264,492,500
  - ts-route-service: cpu P95 ≈ 4.5794, diskio P95 ≈ 64,074
  - ts-travel-service: mem P95 ≈ 276,325,800; socket P95 ≈ 17.0367
  - ts-auth/order/travel/services also have latency-50/90, workload, error, etc. (full table computed)

Anomalies within the incident window (2024-01-23 15:11:51 UTC — 2024-01-23 15:41:51 UTC)
- Top anomaly groups (sorted by anomaly_count):
  1. ts-order-service / error — anomaly_count = 25; earliest anomaly 2024-01-23 15:14:00 UTC; extremal value in window = 0.0
  2. ts-order-service / socket — anomaly_count = 5; earliest anomaly 2024-01-23 15:18:00 UTC; extremal (max) ≈ 13.1167
  3. ts-travel-service / error — anomaly_count = 5; earliest anomaly 2024-01-23 15:34:00 UTC; extremal = 0.0
  4. ts-auth-service / diskio — anomaly_count = 4; earliest anomaly 2024-01-23 15:14:00 UTC; extremal (max) ≈ 163,612.4
  5. ts-auth-service / cpu — anomaly_count = 4; earliest anomaly 2024-01-23 15:16:00 UTC; extremal (max) ≈ 39.79
  6. ts-auth-service / mem — anomaly_count = 4; earliest anomaly 2024-01-23 15:14:00 UTC; extremal (max) ≈ 263,829,100
  7. ts-auth-service / latency-90 — anomaly_count = 4; earliest anomaly 2024-01-23 15:22:00 UTC; extremal (max) ≈ 2.2674
  8. ts-auth-service / latency-50 — anomaly_count = 4; earliest anomaly 2024-01-23 15:22:00 UTC; extremal (max) ≈ 0.8918
  9. ts-order-service / cpu, diskio, latency-50, latency-90, mem, workload — several with anomaly_count = 4 (earliest timestamps ~15:14–15:25 UTC) and respective extremal values reported
 10. ts-route-service / cpu, diskio, latency-50, latency-90, mem — anomaly_count = 4 each (earliest anomalies ~15:14–15:16 UTC)

(Only top 20 groups were included; anomalies defined as value > global P95 OR value <= global P5. If no anomalies existed for a group it was excluded. If no anomalies across all groups an empty summary would be returned.)

Trace and log counts in the incident window (filtered to 5 components)
- trace.csv (windowed counts): many trace_name groups for ts-auth-service and ts-order-service appear with count = 25 (e.g., trace.from_root.*, trace.from_ts-auth-service.*, trace.to_ts-auth-service.*; and similar groups for ts-order-service).
- log.csv (windowed counts): for the four present components (ts-auth-service, ts-order-service, ts-route-service, ts-travel-service) the typical groups log.error_count and log.row_count appear with count = 25 each. ts-train-service is absent from log.csv.

Key takeaways
- ts-order-service shows the most prominent anomaly signal: the 'error' metric flagged across the entire window (25/25 points) and socket spikes (5 anomalies).
- ts-auth-service shows multiple anomalies across disk I/O, CPU, memory, and latency (each with 4 anomaly points, earliest around 15:14–15:22 UTC).
- ts-route-service has several anomalies (cpu, diskio, latency, mem) occurring around 15:14–15:16 UTC.
- ts-travel-service shows 'error' anomalies (5 points) in the window.
- ts-train-service has no log records in log.csv for this dataset (explicitly missing from logs).

If you want, next steps I can do:
- Show the full anomaly summary table (top 20 rows) with exact numeric columns again, or
- Drill into a specific component/kpi (e.g., ts-order-service error/socket or ts-auth-service diskio/cpu) to list the exact timestamps and values in the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  count           p95            p5
0     ts-auth-service         cpu     25  3.973440e+01  1.366078e+01
1     ts-auth-service      diskio     25  1.055891e+05  4.249484e+04
2     ts-auth-service  latency-50     25  8.772784e-01  2.137026e-01
3     ts-auth-service  latency-90     25  2.046220e+00  4.583041e-01
4     ts-auth-service         mem     25  2.634673e+08  2.576922e+08
..                ...         ...    ...           ...           ...
33  ts-travel-service  latency-90     25  2.380152e-01  9.255942e-02
34  ts-travel-service         mem     25  2.763258e+08  2.723057e+08
35  ts-travel-service      socket     25  1.703667e+01  1.381513e+01
36  ts-travel-service    workload     25  1.080247e+01  6.657552e+00
31  ts-travel-service       error      5  0.000000e+00  0.000000e+00

[37 rows x 5 columns],               cmdb_id    kpi_name           p95            p5  anomaly_count earliest_anomaly_timestamp  extremal_value_in_window
0    ts-order-service       error  0.000000e+00  0.000000e+00             25  2024-01-23 15:14:00+00:00              0.000000e+00
1    ts-order-service      socket  1.284667e+01  1.111667e+01              5  2024-01-23 15:18:00+00:00              1.311667e+01
2   ts-travel-service       error  0.000000e+00  0.000000e+00              5  2024-01-23 15:34:00+00:00              0.000000e+00
3     ts-auth-service      diskio  1.055891e+05  4.249484e+04              4  2024-01-23 15:14:00+00:00              1.636124e+05
4     ts-auth-service         cpu  3.973440e+01  1.366078e+01              4  2024-01-23 15:16:00+00:00              3.979049e+01
5     ts-auth-service         mem  2.634673e+08  2.576922e+08              4  2024-01-23 15:14:00+00:00              2.638291e+08
6     ts-auth-service  latency-90  2.046220e+00  4.583041e-01              4  2024-01-23 15:22:00+00:00              2.267387e+00
7     ts-auth-service  latency-50  8.772784e-01  2.137026e-01              4  2024-01-23 15:22:00+00:00              8.917512e-01
8    ts-order-service         cpu  1.871464e+00  1.208375e+00              4  2024-01-23 15:14:00+00:00              1.942378e+00
9    ts-order-service      diskio  6.568335e+04  3.710875e+04              4  2024-01-23 15:15:00+00:00              6.618003e+04
10   ts-order-service  latency-50  7.293611e-03  5.247990e-03              4  2024-01-23 15:25:00+00:00              8.781108e-03
11   ts-order-service  latency-90  7.946036e-02  1.350257e-02              4  2024-01-23 15:17:00+00:00              2.038453e-01
12    ts-auth-service      socket  1.712179e+01  1.165333e+01              4  2024-01-23 15:25:00+00:00              1.938333e+01
13   ts-order-service         mem  2.644925e+08  2.631101e+08              4  2024-01-23 15:14:00+00:00              2.647044e+08
14   ts-order-service    workload  7.673707e+00  4.249942e+00              4  2024-01-23 15:14:00+00:00              7.942183e+00
15   ts-route-service         cpu  4.579391e+00  3.633648e+00              4  2024-01-23 15:16:00+00:00              4.623355e+00
16   ts-route-service      diskio  6.407446e+04  4.061422e+04              4  2024-01-23 15:14:00+00:00              7.110802e+04
17   ts-route-service  latency-50  3.358721e-03  3.211281e-03              4  2024-01-23 15:14:00+00:00              3.363289e-03
18   ts-route-service  latency-90  6.834179e-03  5.023465e-03              4  2024-01-23 15:14:00+00:00              6.874330e-03
19   ts-route-service         mem  2.571501e+08  2.566054e+08              4  2024-01-23 15:14:00+00:00              2.572208e+08,              cmdb_id                                    trace_name  count
0    ts-auth-service                 trace.from_root.duration_mean     25
1    ts-auth-service                  trace.from_root.duration_p95     25
2    ts-auth-service                    trace.from_root.error_rate     25
3    ts-auth-service                     trace.from_root.row_count     25
4    ts-auth-service      trace.from_ts-auth-service.duration_mean     25
5    ts-auth-service       trace.from_ts-auth-service.duration_p95     25
6    ts-auth-service         trace.from_ts-auth-service.error_rate     25
7    ts-auth-service          trace.from_ts-auth-service.row_count     25
8    ts-auth-service        trace.to_ts-auth-service.duration_mean     25
9    ts-auth-service         trace.to_ts-auth-service.duration_p95     25
10   ts-auth-service           trace.to_ts-auth-service.error_rate     25
11   ts-auth-service            trace.to_ts-auth-service.row_count     25
20  ts-order-service     trace.from_ts-order-service.duration_mean     25
21  ts-order-service      trace.from_ts-order-service.duration_p95     25
22  ts-order-service        trace.from_ts-order-service.error_rate     25
23  ts-order-service         trace.from_ts-order-service.row_count     25
32  ts-order-service  trace.from_ts-security-service.duration_mean     25
33  ts-order-service   trace.from_ts-security-service.duration_p95     25
34  ts-order-service     trace.from_ts-security-service.error_rate     25
35  ts-order-service      trace.from_ts-security-service.row_count     25,              cmdb_id         log_name  count
0    ts-auth-service  log.error_count     25
1    ts-auth-service    log.row_count     25
2   ts-order-service  log.error_count     25
3   ts-order-service    log.row_count     25
4   ts-route-service  log.error_count     25
5   ts-route-service    log.row_count     25
6  ts-travel-service  log.error_count     25
7  ts-travel-service    log.row_count     25,       dataset missing_components
0  metric.csv             (none)
1   trace.csv             (none)
2     log.csv   ts-train-service)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing `metric` DataFrame (assumed loaded and timestamp-parsed).
# Define incident window (UTC)
start = pd.to_datetime('2024-01-23 15:11:51', utc=True)
end = pd.to_datetime('2024-01-23 15:41:51', utc=True)

# List of requested pairs
pairs = [
    ('ts-order-service', 'error'),
    ('ts-order-service', 'socket'),
    ('ts-auth-service', 'cpu'),
    ('ts-auth-service', 'diskio'),
    ('ts-auth-service', 'latency-90'),
    ('ts-route-service', 'cpu'),
    ('ts-travel-service', 'error')
]

# Helper to extract windowed rows for a pair (up to 50 rows, ascending timestamp)
def windowed_rows(cmdb_id, kpi_name):
    df = metric[
        (metric['cmdb_id'] == cmdb_id) &
        (metric['kpi_name'] == kpi_name) &
        (metric['timestamp'] >= start) &
        (metric['timestamp'] <= end)
    ].loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value']].sort_values('timestamp').head(50).reset_index(drop=True)
    return df

# Extract for each pair and collect missing info
ts_order_error = windowed_rows('ts-order-service', 'error')
ts_order_socket = windowed_rows('ts-order-service', 'socket')
ts_auth_cpu = windowed_rows('ts-auth-service', 'cpu')
ts_auth_diskio = windowed_rows('ts-auth-service', 'diskio')
ts_auth_latency90 = windowed_rows('ts-auth-service', 'latency-90')
ts_route_cpu = windowed_rows('ts-route-service', 'cpu')
ts_travel_error = windowed_rows('ts-travel-service', 'error')

# Summarize missing pairs explicitly
missing_rows = []
for varname, (cid, kpi) in zip(
    ['ts_order_error','ts_order_socket','ts_auth_cpu','ts_auth_diskio','ts_auth_latency90','ts_route_cpu','ts_travel_error'],
    pairs
):
    df = globals()[varname]
    if df.empty:
        missing_rows.append({'cmdb_id': cid, 'kpi_name': kpi, 'note': 'missing in incident window'})

missing_pairs_summary = pd.DataFrame(missing_rows)

# Return the labeled DataFrames (variable names serve as labels) and missing summary.
ts_order_error, ts_order_socket, ts_auth_cpu, ts_auth_diskio, ts_auth_latency90, ts_route_cpu, ts_travel_error, missing_pairs_summary

```
Out[3]:
```
summary = (
    "Summary for requested metric pairs in incident window (2024-01-23 15:11:51 UTC to 2024-01-23 15:41:51 UTC):\n\n"
    "1) (ts-order-service, error): 25 rows from 2024-01-23 15:14:00 to 2024-01-23 15:38:00 UTC. All values = 0.0.\n\n"
    "2) (ts-order-service, socket): 25 rows from 2024-01-23 15:14:00 to 2024-01-23 15:38:00 UTC. Values ~11.12–13.12; max ≈ 13.116667 at 15:35 UTC, min ≈ 11.116667 at 15:36 UTC.\n\n"
    "3) (ts-auth-service, cpu): 25 rows from 2024-01-23 15:14:00 to 2024-01-23 15:38:00 UTC. Values rose during the window; min ≈ 13.34 (15:16), max ≈ 39.79 (15:37).\n\n"
    "4) (ts-auth-service, diskio): 25 rows from 2024-01-23 15:14:00 to 2024-01-23 15:38:00 UTC. Observed a spike: max ≈ 163,612.39 at 15:15 UTC; other values vary ~47k–117k within the window.\n\n"
    "5) (ts-auth-service, latency-90): 25 rows from 2024-01-23 15:14:00 to 2024-01-23 15:38:00 UTC. Values increased later in the window; max ≈ 2.267387 at 15:34 UTC (earlier ~0.53–0.68).\n\n"
    "6) (ts-route-service, cpu): 25 rows from 2024-01-23 15:14:00 to 2024-01-23 15:38:00 UTC. Values ~3.32–4.62; peak ≈ 4.623355 at 15:16 UTC.\n\n"
    "7) (ts-travel-service, error): 5 rows from 2024-01-23 15:34:00 to 2024-01-23 15:38:00 UTC. All values = 0.0.\n\n"
    "Missing pairs: none — all requested pairs had rows in the incident window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp           cmdb_id kpi_name  value
0  2024-01-23 15:14:00+00:00  ts-order-service    error    0.0
1  2024-01-23 15:15:00+00:00  ts-order-service    error    0.0
2  2024-01-23 15:16:00+00:00  ts-order-service    error    0.0
3  2024-01-23 15:17:00+00:00  ts-order-service    error    0.0
4  2024-01-23 15:18:00+00:00  ts-order-service    error    0.0
..                       ...               ...      ...    ...
20 2024-01-23 15:34:00+00:00  ts-order-service    error    0.0
21 2024-01-23 15:35:00+00:00  ts-order-service    error    0.0
22 2024-01-23 15:36:00+00:00  ts-order-service    error    0.0
23 2024-01-23 15:37:00+00:00  ts-order-service    error    0.0
24 2024-01-23 15:38:00+00:00  ts-order-service    error    0.0

[25 rows x 4 columns],                    timestamp           cmdb_id kpi_name      value
0  2024-01-23 15:14:00+00:00  ts-order-service   socket  12.000000
1  2024-01-23 15:15:00+00:00  ts-order-service   socket  11.533333
2  2024-01-23 15:16:00+00:00  ts-order-service   socket  12.666667
3  2024-01-23 15:17:00+00:00  ts-order-service   socket  12.266667
4  2024-01-23 15:18:00+00:00  ts-order-service   socket  12.866667
..                       ...               ...      ...        ...
20 2024-01-23 15:34:00+00:00  ts-order-service   socket  12.283333
21 2024-01-23 15:35:00+00:00  ts-order-service   socket  13.116667
22 2024-01-23 15:36:00+00:00  ts-order-service   socket  11.116667
23 2024-01-23 15:37:00+00:00  ts-order-service   socket  12.400000
24 2024-01-23 15:38:00+00:00  ts-order-service   socket  12.038462

[25 rows x 4 columns],                    timestamp          cmdb_id kpi_name      value
0  2024-01-23 15:14:00+00:00  ts-auth-service      cpu  15.833876
1  2024-01-23 15:15:00+00:00  ts-auth-service      cpu  15.503167
2  2024-01-23 15:16:00+00:00  ts-auth-service      cpu  13.339966
3  2024-01-23 15:17:00+00:00  ts-auth-service      cpu  14.806135
4  2024-01-23 15:18:00+00:00  ts-auth-service      cpu  15.561155
..                       ...              ...      ...        ...
20 2024-01-23 15:34:00+00:00  ts-auth-service      cpu  37.532033
21 2024-01-23 15:35:00+00:00  ts-auth-service      cpu  38.077316
22 2024-01-23 15:36:00+00:00  ts-auth-service      cpu  39.669252
23 2024-01-23 15:37:00+00:00  ts-auth-service      cpu  39.790490
24 2024-01-23 15:38:00+00:00  ts-auth-service      cpu  39.041485

[25 rows x 4 columns],                    timestamp          cmdb_id kpi_name          value
0  2024-01-23 15:14:00+00:00  ts-auth-service   diskio  117060.243503
1  2024-01-23 15:15:00+00:00  ts-auth-service   diskio  163612.389002
2  2024-01-23 15:16:00+00:00  ts-auth-service   diskio   50560.600738
3  2024-01-23 15:17:00+00:00  ts-auth-service   diskio   52811.954595
4  2024-01-23 15:18:00+00:00  ts-auth-service   diskio   49193.803110
..                       ...              ...      ...            ...
20 2024-01-23 15:34:00+00:00  ts-auth-service   diskio   47966.297903
21 2024-01-23 15:35:00+00:00  ts-auth-service   diskio   42005.678061
22 2024-01-23 15:36:00+00:00  ts-auth-service   diskio   51362.539232
23 2024-01-23 15:37:00+00:00  ts-auth-service   diskio   59348.319813
24 2024-01-23 15:38:00+00:00  ts-auth-service   diskio   51993.503378

[25 rows x 4 columns],                    timestamp          cmdb_id    kpi_name     value
0  2024-01-23 15:14:00+00:00  ts-auth-service  latency-90  0.646482
1  2024-01-23 15:15:00+00:00  ts-auth-service  latency-90  0.675047
2  2024-01-23 15:16:00+00:00  ts-auth-service  latency-90  0.563723
3  2024-01-23 15:17:00+00:00  ts-auth-service  latency-90  0.527687
4  2024-01-23 15:18:00+00:00  ts-auth-service  latency-90  0.605024
..                       ...              ...         ...       ...
20 2024-01-23 15:34:00+00:00  ts-auth-service  latency-90  2.267387
21 2024-01-23 15:35:00+00:00  ts-auth-service  latency-90  1.937774
22 2024-01-23 15:36:00+00:00  ts-auth-service  latency-90  1.960516
23 2024-01-23 15:37:00+00:00  ts-auth-service  latency-90  1.872734
24 2024-01-23 15:38:00+00:00  ts-auth-service  latency-90  2.065730

[25 rows x 4 columns],                    timestamp           cmdb_id kpi_name     value
0  2024-01-23 15:14:00+00:00  ts-route-service      cpu  3.746220
1  2024-01-23 15:15:00+00:00  ts-route-service      cpu  4.140097
2  2024-01-23 15:16:00+00:00  ts-route-service      cpu  4.623355
3  2024-01-23 15:17:00+00:00  ts-route-service      cpu  4.191154
4  2024-01-23 15:18:00+00:00  ts-route-service      cpu  3.777537
..                       ...               ...      ...       ...
20 2024-01-23 15:34:00+00:00  ts-route-service      cpu  3.804290
21 2024-01-23 15:35:00+00:00  ts-route-service      cpu  3.921489
22 2024-01-23 15:36:00+00:00  ts-route-service      cpu  3.996375
23 2024-01-23 15:37:00+00:00  ts-route-service      cpu  3.316379
24 2024-01-23 15:38:00+00:00  ts-route-service      cpu  3.784112

[25 rows x 4 columns],                   timestamp            cmdb_id kpi_name  value
0 2024-01-23 15:34:00+00:00  ts-travel-service    error    0.0
1 2024-01-23 15:35:00+00:00  ts-travel-service    error    0.0
2 2024-01-23 15:36:00+00:00  ts-travel-service    error    0.0
3 2024-01-23 15:37:00+00:00  ts-travel-service    error    0.0
4 2024-01-23 15:38:00+00:00  ts-travel-service    error    0.0, Empty DataFrame
Columns: []
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing `trace`, `start`, and `end` variables from previous steps.
# If not present in the kernel, (re)define the incident window:
try:
    start, end
except NameError:
    start = pd.to_datetime('2024-01-23 15:11:51', utc=True)
    end = pd.to_datetime('2024-01-23 15:41:51', utc=True)

# Substrings indicating inter-service calls
substrs = [
    'to_ts-order-service', 'to_ts-auth-service', 'to_ts-route-service', 'to_ts-travel-service',
    'from_ts-order-service', 'from_ts-auth-service', 'from_ts-route-service', 'from_ts-travel-service'
]
pattern = '|'.join(substrs)

# Filter trace to window and matching trace_name
trace_window = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()
mask = trace_window['trace_name'].str.contains(pattern, na=False)
trace_matches = trace_window[mask].copy()

if trace_matches.empty:
    trace_aggregates = trace_matches  # empty DataFrame
    no_matching_traces = "No matching traces found for the specified substrings in the incident window."
else:
    trace_aggregates = (
        trace_matches
        .groupby(['cmdb_id', 'trace_name'], dropna=False)
        .agg(
            count=('value', 'size'),
            earliest_timestamp=('timestamp', 'min'),
            min_value=('value', 'min'),
            mean_value=('value', 'mean'),
            max_value=('value', 'max')
        )
        .reset_index()
        .sort_values('count', ascending=False)
        .head(20)
        # ensure column order as requested
        [['cmdb_id', 'trace_name', 'count', 'earliest_timestamp', 'min_value', 'mean_value', 'max_value']]
    )
    no_matching_traces = ""

# Return compact results
trace_aggregates, no_matching_traces

```
Out[4]:
```
Summary of matching inter-service traces in the incident window (2024-01-23 15:11:51 — 2024-01-23 15:41:51 UTC):

- Matching traces were found (not empty). Most groups have count = 25 and earliest timestamp = 2024-01-23 15:14:00 UTC.

Top groups (examples, ordered by count):
- root / trace.to_ts-auth-service.duration_mean — count 25; earliest 2024-01-23 15:14:00 UTC; min 0.2325; mean 0.5678; max 1.0410.
- root / trace.to_ts-auth-service.duration_p95 — count 25; earliest 15:14:00; min 0.3591; mean 1.1148; max 2.3627.
- root / trace.to_ts-auth-service.error_rate — count 25; earliest 15:14:00; min/mean/max = 0.0 (no errors recorded in these trace error_rate metrics).
- root / trace.to_ts-auth-service.row_count — count 25; earliest 15:14:00; min 12; mean 78.04; max 89.
- root / trace.to_ts-travel-service.duration_mean — count 25; earliest 15:14:00; min 0.4121; mean 0.5041; max 0.9906.
- root / trace.to_ts-travel-service.duration_p95 — count 25; earliest 15:14:00; min 0.6236; mean 0.7533; max 1.7358.
- root / trace.to_ts-travel-service.error_rate — count 25; earliest 15:14:00; min/mean/max = 0.0.
- root / trace.to_ts-travel-service.row_count — count 25; earliest 15:14:00; min 4; mean 21.32; max 27.
- ts-admin-travel-service / trace.to_ts-travel-service.* groups — each count 25; earliest 15:14:00; row_count min 2 mean 8.8 max 10; duration metrics very small.
- ts-auth-service / trace.from_ts-auth-service.* and ts-auth-service / trace.to_ts-auth-service.* — each count 25; earliest 15:14:00; row_count min 48 mean 311.92 max 352; duration metrics very small on average.

Absent substrings (no matching groups returned in the top results):
- No trace groups matching to/from order-service or to/from route-service were present in the aggregated top results.

Conclusion:
- Inter-service trace activity involving auth-service and travel-service is clearly present and stable (25 samples each) with low error_rate in traces. No matching traces for order-service or route-service were found in these filtered results.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                trace_name  count        earliest_timestamp  min_value  mean_value   max_value
0                      root    trace.to_ts-auth-service.duration_mean     25 2024-01-23 15:14:00+00:00   0.232492    0.567803    1.040964
1                      root     trace.to_ts-auth-service.duration_p95     25 2024-01-23 15:14:00+00:00   0.359112    1.114787    2.362664
2                      root       trace.to_ts-auth-service.error_rate     25 2024-01-23 15:14:00+00:00   0.000000    0.000000    0.000000
3                      root        trace.to_ts-auth-service.row_count     25 2024-01-23 15:14:00+00:00  12.000000   78.040000   89.000000
11                     root      trace.to_ts-travel-service.row_count     25 2024-01-23 15:14:00+00:00   4.000000   21.320000   27.000000
10                     root     trace.to_ts-travel-service.error_rate     25 2024-01-23 15:14:00+00:00   0.000000    0.000000    0.000000
9                      root   trace.to_ts-travel-service.duration_p95     25 2024-01-23 15:14:00+00:00   0.623589    0.753308    1.735751
8                      root  trace.to_ts-travel-service.duration_mean     25 2024-01-23 15:14:00+00:00   0.412052    0.504090    0.990550
15  ts-admin-travel-service      trace.to_ts-travel-service.row_count     25 2024-01-23 15:14:00+00:00   2.000000    8.800000   10.000000
14  ts-admin-travel-service     trace.to_ts-travel-service.error_rate     25 2024-01-23 15:14:00+00:00   0.000000    0.000000    0.000000
13  ts-admin-travel-service   trace.to_ts-travel-service.duration_p95     25 2024-01-23 15:14:00+00:00   0.056597    0.067856    0.144178
12  ts-admin-travel-service  trace.to_ts-travel-service.duration_mean     25 2024-01-23 15:14:00+00:00   0.053301    0.057406    0.077794
16          ts-auth-service  trace.from_ts-auth-service.duration_mean     25 2024-01-23 15:14:00+00:00   0.001386    0.005107    0.013345
17          ts-auth-service   trace.from_ts-auth-service.duration_p95     25 2024-01-23 15:14:00+00:00   0.001671    0.033597    0.095886
18          ts-auth-service     trace.from_ts-auth-service.error_rate     25 2024-01-23 15:14:00+00:00   0.000000    0.000000    0.000000
19          ts-auth-service      trace.from_ts-auth-service.row_count     25 2024-01-23 15:14:00+00:00  48.000000  311.920000  352.000000
23          ts-auth-service        trace.to_ts-auth-service.row_count     25 2024-01-23 15:14:00+00:00  48.000000  311.920000  352.000000
22          ts-auth-service       trace.to_ts-auth-service.error_rate     25 2024-01-23 15:14:00+00:00   0.000000    0.000000    0.000000
21          ts-auth-service     trace.to_ts-auth-service.duration_p95     25 2024-01-23 15:14:00+00:00   0.001671    0.033597    0.095886
20          ts-auth-service    trace.to_ts-auth-service.duration_mean     25 2024-01-23 15:14:00+00:00   0.001386    0.005107    0.013345, '')```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse existing variables if present; otherwise ensure timestamp parsing
try:
    log
except NameError:
    log = pd.read_csv("log.csv")
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Incident window (reuse if defined)
try:
    start, end
except NameError:
    start = pd.to_datetime('2024-01-23 15:11:51', utc=True)
    end = pd.to_datetime('2024-01-23 15:41:51', utc=True)

# Components of interest
components = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Filter logs to components and incident window
log_window = log[
    log['cmdb_id'].isin(components) &
    (log['timestamp'] >= start) &
    (log['timestamp'] <= end)
].copy()

# Aggregates per (cmdb_id, log_name)
if log_window.empty:
    log_aggregates = log_window.iloc[0:0][['cmdb_id', 'log_name', 'count', 'earliest_timestamp', 'min_value', 'mean_value', 'max_value']]
else:
    log_aggregates = (
        log_window
        .groupby(['cmdb_id', 'log_name'], dropna=False)
        .agg(
            count=('value', 'size'),
            earliest_timestamp=('timestamp', 'min'),
            min_value=('value', 'min'),
            mean_value=('value', 'mean'),
            max_value=('value', 'max')
        )
        .reset_index()
        .sort_values(['cmdb_id', 'log_name'], ascending=[True, True])
        .head(20)
    )

# Identify components missing from log_window
present_in_window = set(log_window['cmdb_id'].unique())
missing_components = [c for c in components if c not in present_in_window]
missing_summary = pd.DataFrame([{
    'dataset': 'log.csv (windowed)',
    'missing_components': ', '.join(missing_components) if missing_components else '(none)'
}])

# Return compact results
log_aggregates, missing_summary

```
Out[5]:
```
Summary of log aggregates in the incident window (2024-01-23 15:11:51 — 2024-01-23 15:41:51 UTC):

- ts-auth-service
  - log.error_count: count=25, earliest=2024-01-23 15:14:00 UTC, values all 0 (min=0, mean=0.00, max=0)
  - log.row_count:   count=25, earliest=2024-01-23 15:14:00 UTC, min=48, mean=311.92, max=352

- ts-order-service
  - log.error_count: count=25, earliest=2024-01-23 15:14:00 UTC, values all 0 (min=0, mean=0.00, max=0)
  - log.row_count:   count=25, earliest=2024-01-23 15:14:00 UTC, min=75, mean=356.76, max=478

- ts-route-service
  - log.error_count: count=25, earliest=2024-01-23 15:14:00 UTC, values all 0 (min=0, mean=0.00, max=0)
  - log.row_count:   count=25, earliest=2024-01-23 15:14:00 UTC, min=172, mean=953.64, max=1097

- ts-travel-service
  - log.error_count: count=25, earliest=2024-01-23 15:14:00 UTC, values all 0 (min=0, mean=0.00, max=0)
  - log.row_count:   count=25, earliest=2024-01-23 15:14:00 UTC, min=340, mean=1610.24, max=2029

Missing components in the windowed logs:
- ts-train-service has no log rows in the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  count        earliest_timestamp  min_value  mean_value  max_value
0    ts-auth-service  log.error_count     25 2024-01-23 15:14:00+00:00          0        0.00          0
1    ts-auth-service    log.row_count     25 2024-01-23 15:14:00+00:00         48      311.92        352
2   ts-order-service  log.error_count     25 2024-01-23 15:14:00+00:00          0        0.00          0
3   ts-order-service    log.row_count     25 2024-01-23 15:14:00+00:00         75      356.76        478
4   ts-route-service  log.error_count     25 2024-01-23 15:14:00+00:00          0        0.00          0
5   ts-route-service    log.row_count     25 2024-01-23 15:14:00+00:00        172      953.64       1097
6  ts-travel-service  log.error_count     25 2024-01-23 15:14:00+00:00          0        0.00          0
7  ts-travel-service    log.row_count     25 2024-01-23 15:14:00+00:00        340     1610.24       2029,               dataset missing_components
0  log.csv (windowed)   ts-train-service)```
```